# Lesson 4

Using LLMs to answer questions about a document, using embeddings and vector stores.

In [4]:
# imports

import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

# account for deprecation of LLM model
import datetime

# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

llm_model = "gpt-3.5-turbo"

from langchain_community.vectorstores import Chroma as ch
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from IPython.display import display, Markdown
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [6]:
file = '../.data/OutdoorClothingCatalog_L4.csv'
# initialise CSV loader
loader = CSVLoader(file_path=file)

# Load documents from the documents loader
docs = loader.load()

# print document contents at index 0
print(docs[0])

# Create embeddings
# initalise embeddings
embeddings = OpenAIEmbeddings()

# have a look at what happens when you embed a piece of text
embed = embeddings.embed_query('Hi my name is Harrison')

# looking at the embedding we can see how many different elements there are
print(len(embed))

# we can then look at the first few elements
print(embed[:5])

# Create vector store (modern approach)
# we want to create embeddings for all the text we loaded and store them in a vector store
db = ch.from_documents(docs, embeddings)

# we can use the vector store to find similar pieces of text to incoming queries
query = 'please suggest a shirt with sunblocking'
# we get back a list of documents
docs = db.similarity_search(query)
# we can see the how many documents there are here
print(len(docs))
# have a look at the first document
print(docs[0])

# Create retriever for this vectorstore
retriever = db.as_retriever()

# Define the LLM model
llm = ChatOpenAI(temperature=0, model=llm_model)

# Use a langchain chain to do the retrieval and then do questioning and answering over the retrieved documents.

# Create prompt template
prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:

{context}

Question: {question}
""")

# Create the chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Query
query = "Please list all your shirts with sun protection in a table in markdown and summarize each one."

# Get response
response = chain.invoke(query)

# Display the response
display(Markdown(response))

page_content='name: Women's Campside Oxfords
description: This ultracomfortable lace-to-toe Oxford boasts a broken-in feel right out of the box.' metadata={'source': '../.data/OutdoorClothingCatalog_L4.csv', 'row': 0}
1536
[-0.021964654326438904, 0.006758837960660458, -0.01824948936700821, -0.03923514857888222, -0.014007173478603363]


ImportError: Could not import chromadb python package. Please install it with `pip install chromadb`.

# Embeddings 
Embeddings create numerical representation for pieces of text - captures the semantic meaning of the text. So text with similar content with have similar vectors, meaning we can compare pieces of text in the vector space.

# Vector database
A vector database is a way to store the vector representations that were created in the embeddings. 

We can't pass very large chunks of text into LLMs. So when we get a big incoming document we break it up into smaller chunks of text, enabling us to only pass the most relevant chunks of text to the LLM. We create an embedding for each of these chunks which is then stored in the vector database (this is what happens when we create an index).

Now we have the vector database/index we can use it during runtime to find the pieces of text most relevant to our incoming query. When a query comes in we first create an embedding for the query. We then compare it to the vectors in the vector database and pick the n most similar. These are then returned and based in the prompt to the LLM to get back the final answer.

# Retriever
A retriever is a generic interface that can be underpinned by any methos that takes in a query and returns documents. Vectorstores and embeddings are one such method.

# Stuff method
Stuffing is the simpliest method (most common). You simply stuff all the data into the prompt as context to pass to the language model.
* Pros: It makes a single call to the LLM. The LLM has access to all the data at once.
* Cons: LLMs have a context length and for large documents or many documennts this will not work as it will result in a prompt larger than the context length.

# Other methods
These methods are good for if you want to do lots of questionning and answering over lots of different chunks of text.


Map_reduce (second most common): takes all the chunks and passes them along with the question to the LLM, gets back a response and then uses another language model call to summarise all the individual responses into a final answer. It is commonly used for summarisation - recursively summarise pieces of information in it.
* Pro: Map_reduce is powerful because it can operate over any number of documents and it can do individual questions in parallel. 
* Con: It takes a lot more calls, and treats documents as independent (which may not be the most desired thing).

Refine: loops over many documents iteratively - builds upon the answer from the previous document - so it is good for combining documents and building up an answer over time. Generally leads to longer answers. It is not as fast because the language model calls aren't independent.

Map_rerank: you do a single call to the language model for each document and you ask it to return a score. Then you select the highest score. This relies on the language model knowing what the score should be, meaning you have to tell it that it should be a high score if it's relevant to the document (you need to refine instructions). The calls are independent, meaning they can be done in parralel so it is relatively fast. As you are batching them you are making more LLM calls so it will be more expensive. 




Plan:
* Go through video 
* Copy code into my notebook and get cursor to translate it
* Go through translated text and make notes